In [ ]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

In [ ]:
sns.set_context("paper", font_scale=1.4)


colors = [
 "#394C89",  # true light blue (clean, unambiguous)
 "#86C3FF",  # deeper indigo (clearly blue–purple)
 "#9771CC",  # lavender-gray
 "#DE92AB",  # steel blue (lighter, grayer)
 "#C25A7D",  # soft rose
 '#6F7A8A',  # dark cool gray
 '#B8C0CE',  # light cool gray
 "#6CDFD5",  # fresh turquoise
]


colors_3 = [
    "#394C89", "#93A4DC", "#BFBFBF",
]

colors_lng = [
    '#4E79A7', '#F28E2B', '#59A14F', '#E15759', '#76B7B2', '#9C9C9C']




In [ ]:
base_path = Path("path/to/ICD11_WHO")
results_folder = base_path.joinpath("results_final")
figures_folder = results_folder.joinpath('_Figures', 'mean_std')
figures_folder.mkdir(exist_ok=True)

### Load results

In [ ]:
# Define your mapping
mapping_models = {
    "Qwen25_32B": "Qwen 32B",
    "deepseek_70B": "DeepSeek R1 70B",
    "gemma3": "Gemma 3",
    "llama31_8B": "Llama 3.1 8B",
    "llama33_70B": "Llama 3.3 70B",
    "mistral_7B": "Mistral 7B",
    "mistral_large": "Mistral Large",
}

In [ ]:
files = sorted(sorted(results_folder.glob("*/*/*translated_performance_corrected.csv")) + sorted(results_folder.glob("*/english/*performance_corrected.csv")))

files = [i for i in files if (not "results_old" in str(i))] #(not "english" in str(i)) and 

df_list = []

for file in files:
    if "gpt" in str(file): continue
    df_lng = round(pd.read_csv(file) * 100, 2)
    df_lng["llm"] = file.parts[-3]
    df_lng["language"] = file.parts[-2]
    df_lng = df_lng.set_index(['llm', 'language'])
    df_list.append(df_lng)

df = pd.concat(df_list)
# Rename models for plots
df = df.rename(index=mapping_models, level='llm')
order = [
    'Mistral 7B',
    'Llama 3.1 8B',
    'Gemma 3',
    'Qwen 32B',
    'DeepSeek R1 70B',
    'Llama 3.3 70B',
    'Mistral Large'
]

llm_cat = pd.Categorical(
    df.index.get_level_values('llm'),
    categories=order,
    ordered=True
)

# Use that to reorder the rows (MultiIndex is preserved)
df = df.iloc[llm_cat.argsort(kind='stable')]

df.filter(like="top_1").to_csv(results_folder.joinpath("LLM_performance_multilingual_top1.csv"))

In [ ]:
df_english = df.xs("english",level="language", drop_level=False)
df_english

### Load clinician performance

In [ ]:
df_clinicians = pd.read_excel(results_folder.joinpath('clinicians','Clinician Rating Performance.xlsx'), sheet_name="Overall Performance", index_col='Category')
df_clinicians.loc["mean","Mean"] = df_clinicians["Mean"].mean()
df_clinicians.loc["std","Mean"] = df_clinicians["Mean"].std()
df_clinicians = round(df_clinicians,2)
df_clinicians

In [ ]:
df_comparison = df_english.filter(like='1')
for category in ["Anxiety","Stress", "Mood","mean", "std"]:
    df_comparison.loc[pd.IndexSlice['Clinicians','english'],f'{category}_top_1_accuracy'] = df_clinicians.loc[category, "Mean"]
df_comparison.to_csv(results_folder.joinpath("performance_english.csv"))

### Plot performance of clinicians and LLMs

In [ ]:
colors = [
 "#22A37A",  # lavender-gray
 "#95DEC7", 
 "#3461B9",  # true light blue (clean, unambiguous)
 "#83A1DF",  # deeper indigo (clearly blue–purple)
 # steel blue (lighter, grayer)
 "#6F7A8A",  # soft rose
 "#B0B0B0",  # dark cool gray
 "#F0F0F0",  # light cool gray
 "#E08383",  # fresh turquoise
]

# Reset index in case llm is the index
df_plot = df_comparison.reset_index()

fig, ax = plt.subplots(figsize=(6, 5))

order = df_plot['llm'].tolist()
x = np.arange(len(order))
ax.errorbar(
    x,
    df_plot['mean_top_1_accuracy'].values,
    yerr=df_plot['std_top_1_accuracy'].values,
    fmt='none',
    capsize=4,
    ecolor='black',
    elinewidth=1,
    zorder=3
)

# Draw barplot with error bars from std_top_1_accuracy
sns.barplot(
    data=df_plot,
    x="llm",
    y="mean_top_1_accuracy",
    palette=colors,
    edgecolor="black",
    linewidth=0.8,
    alpha=1.0,      # fully opaque
    ax=ax,
)


# --- Add stripes (hatching) for the clinician bar ---
for i, patch in enumerate(ax.patches):
    model = order[i]
    if model.lower() == "clinicians":     # match name (case-insensitive)
        patch.set_hatch('/')            # diagonal stripes
        patch.set_edgecolor("black")     # keep clear border
        patch.set_linewidth(1.0)
        patch.set_alpha(1.0)   

# Aesthetics
ax.set_title("")
ax.set_xlabel("Model")
ax.set_ylabel("Mean Accuracy (%)")
ax.set_ylim(0, 102)
# Make grid go behind bars
ax.set_axisbelow(True)
ax.grid(axis='y', linestyle='--', alpha=0.4)

# Legend (optional, if you group by something)
# ax.legend(title="Model", frameon=False, loc="upper center", bbox_to_anchor=(0.5, 1.15), ncol=4)

# Rotate labels for readability
plt.xticks(rotation=45, ha='right')
plt.xlabel('')
plt.tight_layout()
plt.savefig(figures_folder.joinpath("English_Performance_meanStd_newColors.pdf"), format='pdf',bbox_inches="tight")
plt.show()


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Define the categories and corresponding column names
categories = ["Anxiety", "Stress", "Mood"]

# Melt into long format for Seaborn
df_long = pd.DataFrame()

for cat in categories:
    mean_col = f"{cat}_top_1_accuracy"
    std_col  = f"{cat}_std_top_1_accuracy"  # adjust if necessary

    tmp = df_plot[["llm", mean_col]].copy()
    tmp = tmp.rename(columns={mean_col: "mean_top_1_accuracy"})
    tmp["category"] = cat

    if std_col in df_plot.columns:
        tmp["std_top_1_accuracy"] = df_plot[std_col]
    else:
        tmp["std_top_1_accuracy"] = np.nan

    df_long = pd.concat([df_long, tmp], ignore_index=True)

# --- Plot ---
fig, ax = plt.subplots(figsize=(10, 6))

sns.barplot(
    data=df_long,
    x="category",
    y="mean_top_1_accuracy",
    hue="llm",
    palette=colors,
    errorbar=None,       # we'll add our own error bars
    edgecolor="black",
    linewidth=0.8,
    alpha=1.0,
    ax=ax,
)

# --- Add stripes (hatching) for the clinician bar ---
df_sorted = df_long.reset_index(drop=True)
# For each category (x=0,1,2), find the rightmost bar and hatch it
for x_center in [0, 1, 2]:  # one per category
    # find patches whose center is closest to this category (within ±0.4)
    patches_in_cat = [p for p in ax.patches if abs((p.get_x() + p.get_width()/2) - x_center) < 0.5]
    if not patches_in_cat:
        continue
    # clinician bar is the one with largest x (rightmost within group)
    clinician_patch = max(patches_in_cat, key=lambda p: p.get_x())
    clinician_patch.set_hatch('/')            # diagonal stripes
    clinician_patch.set_edgecolor("black")     # keep clear border
    clinician_patch.set_linewidth(1.0)
    clinician_patch.set_alpha(1.0)  

# Add upper error bars manually
# Get positions for each group of bars
group_width = 0.8
n_hue = len(df_long["llm"].unique())
bar_width = group_width / n_hue

# --- Aesthetics ---
ax.set_title("")
ax.set_xlabel("")
ax.set_ylabel("Accuracy (%)")
ax.set_ylim(0, 105)
ax.set_axisbelow(True)
ax.grid(axis='y', linestyle='--', alpha=0.4)

# Legend on top
ax.legend(
    title="Model",
    loc="upper center",
    bbox_to_anchor=(0.5, 1.3),
    frameon=True,
    ncol=min(len(df_plot["llm"]), 4)
)

plt.tight_layout()
plt.savefig(figures_folder.joinpath("English_Performance_perCategory_newColors.pdf"), format='pdf',bbox_inches="tight")
plt.show()


### Cohen's kappa

In [ ]:
colors = [
 "#22A37A",  # lavender-gray
 "#95DEC7", 
 "#3461B9",  # true light blue (clean, unambiguous)
 "#83A1DF",  # deeper indigo (clearly blue–purple)
 # steel blue (lighter, grayer)
 "#6F7A8A",  # soft rose
 "#B0B0B0",  # dark cool gray
 "#F0F0F0",  # light cool gray
 "#E08383",  # fresh turquoise
]


df_cappa = pd.read_excel(results_folder.joinpath("CohensKappa_LLM_vs_ClinicianConsensus.xlsx"), sheet_name='MeanStd_ByModel')
df_cappa = df_cappa.replace(mapping_models)

order = [
    'Mistral 7B',
    'Llama 3.1 8B',
    'Gemma 3',
    'Qwen 32B',
    'DeepSeek R1 70B',
    'Llama 3.3 70B',
    'Mistral Large'
]

# Reset index in case llm is the index
df_plot = df_cappa.copy()

df_plot["Model"] = pd.Categorical(
    df_plot["Model"],
    categories=order,
    ordered=True
)

df_plot = df_plot.sort_values("Model")

fig, ax = plt.subplots(figsize=(5.5, 5))

order = df_plot['Model'].tolist()
x = np.arange(len(order))
ax.errorbar(
    x,
    df_plot['Mean_Kappa'].values,
    yerr=df_plot['Std_Kappa'].values,
    fmt='none',
    capsize=4,
    ecolor='black',
    elinewidth=1,
    zorder=3
)

# Draw barplot with error bars from std_top_1_accuracy
sns.barplot(
    data=df_plot,
    x="Model",
    y="Mean_Kappa",
    palette=colors,
    edgecolor="black",
    linewidth=0.8,
    alpha=1.0,      # fully opaque
    ax=ax,
)

# Aesthetics
ax.set_title("")
ax.set_xlabel("Model")
ax.set_ylabel("Mean Cohen's Kappa")
# ax.set_ylim(0, 105)
# Make grid go behind bars
ax.set_axisbelow(True)
ax.grid(axis='y', linestyle='--', alpha=0.4)

# Legend (optional, if you group by something)
# ax.legend(title="Model", frameon=False, loc="upper center", bbox_to_anchor=(0.5, 1.15), ncol=4)

# Rotate labels for readability
plt.xticks(rotation=45, ha='right')
plt.xlabel('')
plt.tight_layout()
plt.savefig(figures_folder.joinpath("English_cohensKappa_meanStd_newColors.pdf"), format='pdf',bbox_inches="tight")
plt.show()


In [ ]:
# Define metrics and corresponding std columns
metrics_topn = ['mean_top_1_accuracy', 'mean_top_2_accuracy', 'mean_top_3_accuracy']
std_topn     = ['std_top_1_accuracy',  'std_top_2_accuracy',  'std_top_3_accuracy']
metric_labels = ['Top-1', 'Top-2', 'Top-3']

# Filter English only
df_eng = df_english.copy()
models = df_eng.index.get_level_values(level='llm').unique()

# Prepare figure
fig, ax = plt.subplots(figsize=(10, 4))

# X positions for models
x = np.arange(len(models))
bar_width = 0.2

# Plot each Top-n as a separate bar series
for j, (metric, std_metric) in enumerate(zip(metrics_topn, std_topn)):
    # Extract means and stds for each model
    yvals = [df_eng.xs(model, level='llm')[metric].iloc[0] for model in models]
    yerrs = [df_eng.xs(model, level='llm')[std_metric].iloc[0] for model in models]

    # Offset bars for each Top-n
    x_positions = x + (j - 1) * bar_width  # centers 3 bars per model

    ax.bar(
        x_positions,
        yvals,
        yerr=yerrs,
        capsize=4,               # small caps on error bars
        width=bar_width,
        label=metric_labels[j],
        color=colors_3[j % len(colors_3)],
        edgecolor='black',
        alpha=0.85,
        linewidth=0.8
    )

# Style
ax.set_title("", fontsize=13)
ax.set_ylabel("Accuracy (%)")
ax.set_xticks(x)
ax.set_xticklabels(models, rotation=30, ha='right')
ax.set_ylim(0, 105)
ax.set_axisbelow(True)
ax.grid(axis='y', linestyle='--', alpha=0.4)

# Legend for Top-n levels
ax.legend(
    title="Top-n Metric",
    ncol=1,
    frameon=True,
    loc='upper center',
    bbox_to_anchor=(1.1, 0.75)
)

plt.tight_layout()
plt.savefig(figures_folder.joinpath("English_Performance_top-n_meanStd.pdf"), format='pdf', bbox_inches="tight")
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Define categories and metrics
categories = ['Anxiety', 'Stress', 'Mood']
metrics_suffixes = ['top_1_accuracy', 'top_2_accuracy', 'top_3_accuracy']
metric_labels = ['Top-1', 'Top-2', 'Top-3']

# Filter English only
df_eng = df_english.copy()
models = df_eng.index.get_level_values(level='llm').unique()

# Prepare figure with 3 rows (one per category)
fig, axes = plt.subplots(3, 1, figsize=(6, 9), sharex=True)

bar_width = 0.2  # width per bar

for j, cat in enumerate(categories):
    ax = axes[j]

    # Extract data for this category
    cols = [f"{cat}_{suffix}" for suffix in metrics_suffixes]
    # skip if any columns missing
    if not all(c in df_eng.columns for c in cols):
        continue

    # X positions for each model
    x = np.arange(len(models))

    # Plot Top-n bars per model
    for k, metric in enumerate(cols):
        yvals = [df_eng.xs(model, level='llm')[metric].iloc[0] for model in models]
        x_positions = x + (k - 1) * bar_width  # centers the 3 bars per model
        ax.bar(
            x_positions,
            yvals,
            width=bar_width,
            label=metric_labels[k],
            color=colors_3[k % len(colors_3)],
            edgecolor='black',
            alpha=0.85
        )

    # Styling per category
    ax.set_title(cat, fontsize=12)
    ax.set_xticks(x)
    ax.set_xticklabels(models, rotation=30, ha='right')
    ax.set_ylim(0, 105)
    ax.set_ylabel("Accuracy (%)")
    ax.set_axisbelow(True)
    ax.grid(axis='y', linestyle='--', alpha=0.4)
    if j == 1:  # middle plot
        ax.set_ylabel("Accuracy (%)")

# Shared legend
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles, labels,
    title="Metric",
    ncol=1,
    frameon=True,
    loc='upper center',
    bbox_to_anchor=(1.08, 1.0)
)

fig.suptitle("", fontsize=13)
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig(figures_folder.joinpath("English_Performance_top-n_perCategory.pdf"), format='pdf', bbox_inches="tight")
plt.show()


## Multi-lingual

In [ ]:
# --- Setup -------------------------------------------------------------------
languages = ['english', 'french', 'spanish', 'chinese', 'japanese', 'russian']
models = [
    'Mistral 7B',
    'Llama 3.1 8B',
    'Gemma 3',
    'Qwen 32B',
    'DeepSeek R1 70B',
    'Llama 3.3 70B',
    'Mistral Large'
 ]

colors_lng = ["#6883CD",  # lighter blue
 "#835FC2",
 "#C0A4EF",
 "#D5D5D5",  # lighter teal
 "#797979",

 '#7FBF97',  # lighter green
 "#11853E",  # lighter red
  "#437AC2",  # lighter amber
 
 ]  # lighter gray
 # light neutral gray
 # neutral gray
 # refined neutral gray


# --- Prepare data ------------------------------------------------------------
df_base = df.reset_index().dropna(subset=['language', 'llm'])

x = np.arange(len(models))
group_width = 0.7
bar_w = group_width / max(len(languages), 1)

mean_col, std_col = "mean_top_1_accuracy", "std_top_1_accuracy"

# --- Plot --------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(13, 4))

for j, lang in enumerate(languages):
    y_means, y_stds = [], []

    for model_name in models:
        sub = df_base[(df_base['llm'] == model_name) & (df_base['language'] == lang)]
        if not sub.empty:
            y_means.append(sub[mean_col].iloc[0] if mean_col in sub.columns else np.nan)
            y_stds.append(sub[std_col].iloc[0] if std_col in sub.columns else np.nan)
        else:
            y_means.append(np.nan)
            y_stds.append(np.nan)

    offset = (j - (len(languages) - 1) / 2) * bar_w

    # Bars
    ax.bar(
        x + offset,
        y_means,
        width=bar_w,
        label=str(lang).capitalize(),
        color=colors_lng[j % len(colors_lng)],
        edgecolor="black",
        linewidth=0.8,
        zorder=2
    )

    # Symmetric std error bars (± std)
    ax.errorbar(
        x + offset,
        y_means,
        yerr=y_stds,
        fmt="none",
        ecolor="black",
        elinewidth=1,
        capsize=4,
        capthick=1,
        zorder=3
    )

# --- Formatting ---------------------------------------------------------------
ax.set_title("")
ax.set_xlabel("")
ax.set_ylabel("Accuracy (%)")

# Shift x labels slightly left (optional aesthetic tweak)
tick_shift = -group_width / 2 + bar_w / 2
ax.set_xticks(x)
ax.set_xticklabels(models)

ax.set_ylim(0, 105)
ax.set_axisbelow(True)
ax.grid(axis='y', linestyle='--', alpha=0.4)

# --- Legend on top (centered horizontally) ---
handles, labels = ax.get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    title="Language",
    loc="upper center",
    bbox_to_anchor=(0.525, 1.25),
    ncol=len(labels) // 2 or 1
)

plt.tight_layout()
plt.savefig(
    figures_folder.joinpath("Multilingual_Performance_meanSt_newColors.pdf"),
    format='pdf',
    bbox_inches="tight"
)
plt.show()


In [ ]:
# --- Setup: same palettes & helpers you already had ---------------------------------
metrics_top1 = ['Anxiety_top_1_accuracy', 'Mood_top_1_accuracy', 'Stress_top_1_accuracy']

def prettify_metric(m):
    return (m.replace('_', ' ')
             .replace('top 1', 'Top-1')
             .replace('top 2', 'Top-2')
             .replace('top 3', 'Top-3')
           )

# Work from a base frame that has language + llm; keep potential metric NaNs
df_base = df.reset_index().dropna(subset=['language', 'llm'])

languages = ['english', 'french', 'spanish', 'chinese', 'japanese', 'russian']
models = df_base['llm'].unique().tolist()

x = np.arange(len(models))  # now x represents models
group_width = 0.7
bar_w = group_width / max(len(languages), 1)

# --- Helper to get mean/std columns for Top-n ----------------------------------------
def metrics_for_topn(n):
    return f"mean_top_{n}_accuracy", f"std_top_{n}_accuracy"

# --- Plot: 1 row × 3 subplots for Top-1/Top-2/Top-3 ---------------------------------
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharey=True)
top_ns = [1, 2, 3]

for ax, n in zip(axes, top_ns):
    mean_col, std_col = metrics_for_topn(n)

    # --- Plot bars for each language (hue) ---
    for j, lang in enumerate(languages):
        y_means, y_stds = [], []

        for model_name in models:
            sub = df_base[(df_base['llm'] == model_name) & (df_base['language'] == lang)]
            if not sub.empty:
                y_means.append(sub[mean_col].iloc[0] if mean_col in sub.columns else np.nan)
                y_stds.append(sub[std_col].iloc[0] if std_col in sub.columns else np.nan)
            else:
                y_means.append(np.nan)
                y_stds.append(np.nan)

        # Horizontal offset for each language
        offset = (j - (len(languages) - 1) / 2) * bar_w

        # Bars
        ax.bar(
            x + offset,
            y_means,
            width=bar_w,
            label=str(lang).capitalize(),
            color=colors_lng[j % len(colors)],
            edgecolor="black",
            linewidth=0.8,
            zorder=2
        )

        # --- Symmetric ±std error bars ---
        ax.errorbar(
            x + offset,
            y_means,
            yerr=y_stds,
            fmt="none",
            ecolor="black",
            elinewidth=1,
            capsize=4,
            capthick=1,
            zorder=3
        )

    # --- Axes formatting ---
    ax.set_title(f"Top-{n} Mean Accuracy (Anxiety / Mood / Stress)")
    ax.set_xlabel("")
    ax.set_xticks(x)
    ax.set_xticklabels(models)
    ax.set_axisbelow(True)
    ax.grid(axis='y', linestyle='--', alpha=0.4)
    ax.set_ylabel("Accuracy (%)")

# Shared y-axis label, legend, and limits
for ax in axes:
    ax.set_ylim(0, 105)

# Shared legend (now showing languages)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles, labels,
    title="Language",
    ncol=min(len(languages), 3),
    frameon=True,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.07)
)

fig.suptitle("Top-n Mean Accuracies with Standard Deviation Across Categories",
             y=1.12, fontsize=14)

plt.tight_layout()
plt.savefig(
    figures_folder.joinpath("Multilingual_Performance_top-n_meanStd.pdf"),
    format='pdf',
    bbox_inches="tight"
)
plt.show()


In [ ]:
metrics = ['Anxiety_top_1_accuracy', 'Mood_top_1_accuracy', 'Stress_top_1_accuracy']
df_reset = df[metrics].reset_index().dropna(subset=['language','llm'])  # keep rows with language/model

languages = ['english','french','spanish', 'chinese','japanese','russian']
models    = df_reset['llm'].unique().tolist()

x = np.arange(len(models))  # now x = models
group_width = 0.7
bar_w = group_width / len(languages)

def prettify_metric(m):
    return (m.replace('_', ' ')
             .replace('top 1', 'Top-1')
             .replace('top 2', 'Top-2')
             .replace('top 3', 'Top-3')
           )

# --- Create one figure with multiple subplots ---
fig, axes = plt.subplots(len(metrics), 1,  figsize=(12, 10), sharey=True)

for ax, metric in zip(axes, metrics):
    ys = []
    for lang in languages:
        sub = df_reset[df_reset['language'] == lang].set_index('llm')
        vals = [sub[metric].get(m, np.nan) for m in models]
        ys.append(vals)

    # --- Plot grouped bars (hue = language) ---
    for i, (lang, yvals) in enumerate(zip(languages, ys)):
        offset = (i - (len(languages)-1)/2) * bar_w
        ax.bar(
            x + offset,
            yvals,
            width=bar_w,
            label=str(lang).capitalize(),
            color=colors_lng[i % len(colors)],
            edgecolor="black",
            linewidth=0.8,
        )

    # --- Aesthetics ---
    ax.set_title(prettify_metric(metric))
    ax.set_xlabel("")
    ax.set_xticks(x)
    ax.set_xticklabels(models) #, rotation=30, ha='right')
    ax.set_ylim(0, 105)
    ax.set_axisbelow(True)
    ax.grid(axis='y', linestyle='--', alpha=0.4)
    ax.set_ylabel("Accuracy (%)")


fig.legend(handles, labels, title="Language", ncol=min(len(languages), 3),
           frameon=True, loc="upper center", bbox_to_anchor=(0.52, 1.1))

# fig.suptitle("Mean Accuracies Across Categories", y=1.12, fontsize=14)

plt.tight_layout()

plt.savefig(
    figures_folder.joinpath("Multilingual_Performance_perCategory.pdf"),
    format='pdf',
    bbox_inches="tight"
)
plt.show()
